# Solutions: Refactor monolithic code to modular code (imputation example)

This notebook provides step-by-step solutions for refactoring a monolithic imputation script into modular code, following RAP best practices. Each step matches the corresponding exercise notebook.

Run the following code to run the solutions code snippets:

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..", "..", "src")))

from python_rap_demo.cleaning import clean_health_data
from python_rap_demo.io import read_health_data
from python_rap_demo.processing import calculate_disease_prevalence
from python_rap_demo.utils import add_bmi_column

input_path = "../../data/input/health_data.csv"
cleaned_path = "../outputs/02_modules/cleaned_data.csv"
report_path = "../outputs/02_modules/"

# I/O: Read data
df = read_health_data(input_path)

# Cleaning
df_clean = clean_health_data(df)
df_clean = add_bmi_column(df_clean)

# Processing
prevalence_df = calculate_disease_prevalence(df_clean)

## Step 1 solution: Imputation function

Below is a function that performs imputation for missing height and weight values using group means (e.g., by gender or diagnosis), and flags which values were imputed. This approach is more robust and transparent than using overall means.

In [ ]:
# Modular solution using subfunctions
import pandas as pd


def flag_missing(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Add boolean columns to flag missing values for specified columns.
    """
    df = df.copy()
    for col in columns:
        df[f"{col}_imputed"] = df[col].isna()
    return df


def impute_by_group(df: pd.DataFrame, col: str, group_col: str) -> pd.Series:
    """
    Impute missing values in a column using group means, fallback to overall mean if needed.
    Rounds imputed values to 1 decimal place for consistency.
    """
    group_means = df.groupby(group_col)[col].transform("mean")
    result = df[col].fillna(group_means)
    result = result.fillna(df[col].mean())
    return result.round(1)


def impute_height_weight(df: pd.DataFrame, group_col: str = "gender") -> pd.DataFrame:
    """
    Impute missing height and weight using group means, flag imputed values (modular).
    """
    df = flag_missing(df, ["height_cm", "weight_kg"])
    df["height_cm"] = impute_by_group(df, "height_cm", group_col)
    df["weight_kg"] = impute_by_group(df, "weight_kg", group_col)

    return df


# Example usage:
imputed = impute_height_weight(df, group_col="gender")
print(imputed.head())

## Step 2 solution: Add functions to pipeline
Where you place these functions depends on the context and what makes most sense for your pipeline. However, for this exercise one appropriate solution is outlined below:

1. Place the `flag_missing`, `impute_by_group` and `impute_height_weight` function into `src/python_rap_demo/cleaning.py`. You could also have added `flag_missing` to utils as it could be used across other modules.

In [ ]:
"""
cleaning.py: Data cleaning functions
"""

import pandas as pd


def clean_health_data(df: pd.DataFrame) -> pd.DataFrame:
    """
    Clean health data by dropping rows with missing values in key columns.

    Args:
        df (pd.DataFrame): Raw health data.

    Returns:
        pd.DataFrame: Cleaned health data with no missing values in critical columns.
    """
    df = df.copy()

    ## Only drop na values in diagnosis ##
    df = df.dropna(subset=["diagnosis"])

    # Fill missing smoker values with 'No'
    df["smoker"] = df["smoker"].fillna("No")

    # Ensure gender is uppercase
    df["gender"] = df["gender"].str.upper()

    return df


def flag_missing(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """
    Add boolean columns to flag missing values for specified columns.
    """
    df = df.copy()
    for col in columns:
        df[f"{col}_imputed"] = df[col].isna()
    return df


def impute_by_group(df: pd.DataFrame, col: str, group_col: str) -> pd.Series:
    """
    Impute missing values in a column using group means, fallback to overall mean if needed.
    Rounds imputed values to 1 decimal place for consistency.
    """
    group_means = df.groupby(group_col)[col].transform("mean")
    result = df[col].fillna(group_means)
    result = result.fillna(df[col].mean())
    return result.round(1)


def impute_height_weight(df: pd.DataFrame, group_col: str = "gender") -> pd.DataFrame:
    """
    Impute missing height and weight using group means, flag imputed values (modular).
    """
    df = flag_missing(df, ["height_cm", "weight_kg"])
    df["height_cm"] = impute_by_group(df, "height_cm", group_col)
    df["weight_kg"] = impute_by_group(df, "weight_kg", group_col)
    return df

2. Add the functions inside the `clean_health_data` function:
    - Add the `group_col` parameter to the clean health data function
    - Make sure to remove height_cm and weight_kg from the dropna subset.

In [ ]:
def clean_health_data(df: pd.DataFrame, group_col: str) -> pd.DataFrame:
    """
    Clean health data by dropping rows with missing values in key columns.

    Args:
        df (pd.DataFrame): Raw health data.
        group_col (str) : The column to group on for mean imputation

    Returns:
        pd.DataFrame: Cleaned health data with no missing values in critical columns.
    """
    df = df.copy()

    ## Only drop na values in diagnosis ##
    df = df.dropna(subset=["diagnosis"])

    # Fill missing smoker values with 'No'
    df["smoker"] = df["smoker"].fillna("No")

    # Ensure gender is uppercase
    df["gender"] = df["gender"].str.upper()

    # Impute mean values per group for height and weight
    imputed = impute_height_weight(df, group_col="gender")

    return imputed


# Example usage
print("Missing values before imputation:")
print(df.isnull().sum())
imputed = clean_health_data(df, group_col="gender")
print("Missing values after imputation:")
print(imputed.isnull().sum())


The functions will then be imported inside `clean_health_data` at the top of main.py with:

```python
from python_rap_demo.cleaning import clean_health_data
```

## Step 3 Solution: Reflection

- Group-based imputation preserves important differences in the data and improves reproducibility.
- Flagging imputed values helps with transparency and downstream analysis.
- Handling edge cases (e.g., all values missing in a group) ensures robustness.

**Extension:**
The above could be taken further to automatically detect categorical and numerical columns, applying imputation to each.  
Look at the example below:


In [ ]:
# for utils.py


def is_numeric_column(series: pd.Series) -> bool:
    """
    Determine if a pandas Series is numeric.

    Args:
        series (pd.Series): Input Series.

    Returns:
        bool: True if numeric, else False.
    """
    return pd.api.types.is_numeric_dtype(series)


def is_categorical_column(series: pd.Series) -> bool:
    """
    Determine if a pandas Series is categorical (object, category, or bool).

    Args:
        series (pd.Series): Input Series.

    Returns:
        bool: True if categorical, else False.
    """
    return (
        pd.api.types.is_object_dtype(series)
        or pd.api.types.is_categorical_dtype(series)
        or pd.api.types.is_bool_dtype(series)
    )


def get_column_types(df: pd.DataFrame) -> tuple[list[str], list[str]]:
    """
    Automatically determine numeric and categorical columns in a DataFrame.

    Args:
        df (pd.DataFrame): Input DataFrame.

    Returns:
        tuple: (numeric_cols, categorical_cols)
    """
    numeric_cols = [col for col in df.columns if is_numeric_column(df[col])]
    categorical_cols = [col for col in df.columns if is_categorical_column(df[col])]
    return numeric_cols, categorical_cols

In [ ]:
# for cleaning.py


from typing import List

# import utils functions above using
# from python_rap_demo.utils import (
#     get_column_types,
#     is_categorical_column,
#     is_numeric_column,
# )


def flag_missing(df: pd.DataFrame, columns: List[str]) -> pd.DataFrame:
    """
    Add boolean columns to flag missing values for specified columns.

    Args:
        df (pd.DataFrame): Input DataFrame.
        columns (List[str]): List of columns to flag.

    Returns:
        pd.DataFrame: DataFrame with new _imputed columns.
    """
    df = df.copy()
    for col in columns:
        df[f"{col}_imputed"] = df[col].isna()
    return df


def impute_numeric_by_group(df: pd.DataFrame, col: str, group_col: str) -> pd.Series:
    """
    Impute missing numeric values using group mean, fallback to overall mean.

    Args:
        df (pd.DataFrame): Input DataFrame.
        col (str): Numeric column to impute.
        group_col (str): Column to group by.

    Returns:
        pd.Series: Imputed column.
    """
    group_means = df.groupby(group_col)[col].transform("mean")
    result = df[col].fillna(group_means)
    result = result.fillna(df[col].mean())
    return result.round(1)


def impute_categorical_by_group(df: pd.DataFrame, col: str, group_col: str) -> pd.Series:
    """
    Impute missing categorical values using group mode, fallback to overall mode.

    Args:
        df (pd.DataFrame): Input DataFrame.
        col (str): Categorical column to impute.
        group_col (str): Column to group by.

    Returns:
        pd.Series: Imputed column.
    """
    group_modes = df.groupby(group_col)[col].transform(
        lambda x: x.mode().iloc[0] if not x.mode().empty else pd.NA
    )
    result = df[col].fillna(group_modes)
    overall_mode = df[col].mode()
    if not overall_mode.empty:
        result = result.fillna(overall_mode.iloc[0])
    return result


def impute_columns(df: pd.DataFrame, group_col: str = "gender") -> pd.DataFrame:
    """
    Impute missing values for both numeric and categorical columns using group statistics.
    Automatically detects column types.

    Args:
        df (pd.DataFrame): Input DataFrame.
        group_col (str): Column to group by for imputation.

    Returns:
        pd.DataFrame: DataFrame with imputed values and flags.
    """
    numeric_cols, categorical_cols = get_column_types(df)

    # Exclude group_col from imputation and flagging
    cols_to_flag = [col for col in numeric_cols + categorical_cols if col != group_col]
    df = flag_missing(df, cols_to_flag)
    for col in numeric_cols:
        if col != group_col:
            df[col] = impute_numeric_by_group(df, col, group_col)
    for col in categorical_cols:
        if col != group_col:
            df[col] = impute_categorical_by_group(df, col, group_col)

    return df

In [ ]:
# Example usage: Automatically impute missing values in health data
imputed = impute_columns(df, group_col="gender")
print(imputed.head())

## Step 4 Solution: Refactor Visualisation Code

Below are example functions for visualising missing values and disease prevalence using Plotly. These would go in `src/python_rap_demo/report.py`.

1. Create the functions

In [ ]:
import plotly.express as px


def plot_missing_values(df: pd.DataFrame, output_path: str) -> None:
    """
    Plot number of missing values per column and save as PNG.
    """
    print(f"building {output_path}missing_values.png")
    missing_counts = df.isnull().sum().reset_index()
    missing_counts.columns = ["column", "missing_count"]
    fig = px.bar(
        missing_counts,
        x="column",
        y="missing_count",
        title="Number of Missing Values per Column",
        labels={"missing_count": "Missing Count", "column": "Column"},
        color_discrete_sequence=["#C44E52"],
    )
    fig.write_image(f"{output_path}missing_values.png")


def plot_disease_prevalence(df: pd.DataFrame, output_path: str, month: str) -> None:
    """
    Plot disease prevalence rates per diagnosis for a given month and save as PNG.

    Args:
        df (pd.DataFrame): DataFrame with columns 'diagnosis' and 'prevalence_rate'.
        output_path (str): Path to output directory.
        month (str): Month string for filename.
    """
    print(f"building {output_path}prevalence_{month}.png")
    fig = px.bar(
        df,
        x="diagnosis",
        y="prevalence_rate",
        title=f"Disease Prevalence for {month}",
        color_discrete_sequence=["#4C72B0"],
    )
    fig.write_image(f"{output_path}prevalence_{month}.png")

2. Add the functions to report.py and import plotly

In [ ]:
"""
report.py: Markdown report generation for RAP pipeline
"""

import pandas as pd

# Import plotly
import plotly.express as px


def format_month_section(month: str, month_df: pd.DataFrame) -> str:
    """
    Format the markdown section for a single month.

    Args:
        month (str): Month string.
        month_df (pd.DataFrame): DataFrame filtered for the month.

    Returns:
        str: Markdown string for the month section.
    """
    lines = [f"## Month: {month}\n"]
    for _, row in month_df.iterrows():
        lines.append(
            f"- {row['diagnosis']}: {row['prevalence_rate']:.2%} "
            f"({int(row['case_count'])} cases)\n"
        )
    lines.append("\n")
    return "".join(lines)


def generate_markdown_report(prevalence_df: pd.DataFrame, output_path: str) -> None:
    """
    Generate a markdown report of disease prevalence rates per month.

    Args:
        prevalence_df (pd.DataFrame): DataFrame with prevalence rates.
        output_path (str): Path to output markdown file.
    """
    with open(output_path, "w") as f:
        f.write("# Disease Prevalence Report\n\n")
        for month in prevalence_df["month"].unique():
            month_df = prevalence_df[prevalence_df["month"] == month]
            f.write(format_month_section(month, month_df))


# Add the new functions to the cleaning module
def plot_missing_values(df: pd.DataFrame, output_path: str) -> None:
    """
    Plot number of missing values per column and save as PNG.
    """
    print(f"building {output_path}missing_values.png")
    missing_counts = df.isnull().sum().reset_index()
    missing_counts.columns = ["column", "missing_count"]
    fig = px.bar(
        missing_counts,
        x="column",
        y="missing_count",
        title="Number of Missing Values per Column",
        labels={"missing_count": "Missing Count", "column": "Column"},
        color_discrete_sequence=["#C44E52"],
    )
    fig.write_image(f"{output_path}missing_values.png")


def plot_disease_prevalence(df: pd.DataFrame, output_path: str, month: str) -> None:
    """
    Plot disease prevalence rates per diagnosis for a given month and save as PNG.

    Args:
        df (pd.DataFrame): DataFrame with columns 'diagnosis' and 'prevalence_rate'.
        output_path (str): Path to output directory.
        month (str): Month string for filename.
    """
    print(f"building {output_path}prevalence_{month}.png")
    fig = px.bar(
        df,
        x="diagnosis",
        y="prevalence_rate",
        color="diagnosis",
        title=f"Disease Prevalence for {month}",
        labels={"prevalence_rate": "Prevalence Rate", "diagnosis": "Diagnosis"},
        color_discrete_sequence=["#4C72B0", "#55A868", "#C44E52"],
    )
    fig.write_image(f"{output_path}prevalence_{month}.png")

3. Plot missing values requires the raw data, so add it as a parameter to generate markdown in `report.py` and `main.py`

In [ ]:
# report.py


import time


def generate_markdown_report(
    prevalence_df: pd.DataFrame, raw_df: pd.DataFrame, output_path: str  # Add the raw_df parameter
) -> None:
    """
    Generate a markdown report of disease prevalence rates per month.

    Args:
        prevalence_df (pd.DataFrame): DataFrame with prevalence rates.
        raw_df (pd.DataFrame): Raw DataFrame with missing values  # Add the raw_df parameter
        output_path (str): Path to output markdown file.
    """
    report_path = f"{output_path}disease_prevalence_report.md"

    with open(report_path, "w") as f:
        f.write("# Disease Prevalence Report\n\n")
        # Create charts
        plot_missing_values(raw_df, output_path)
        # Add image links for the charts
        f.write("## Missing Values Chart\n")
        f.write("![Missing Values](missing_values.png)\n\n")

        for month in prevalence_df["month"].unique():
            month_df = prevalence_df[prevalence_df["month"] == month]
            f.write(format_month_section(month, month_df))

            plot_disease_prevalence(month_df, output_path, month)
            f.write("## Disease Prevalence Chart\n")
            f.write(f"![Disease Prevalence](prevalence_{month}.png)\n\n")

            # To limit resource use
            time.sleep(0.5)

In [ ]:
# main.py


# Reporting
generate_markdown_report(prevalence_df, df, report_path)